# Replay BO with fixed gpax backend

Takes the observations (`df_results`) collected during the live run and re-runs the BO acquisition pipeline **offline** — i.e. no microscope, no imaging, just GP fitting + acquisition selection on the already-measured FOVs.

The fix: EI reference point is now the **GP-predicted best marginal mean** over the control grid instead of the raw observed max, so EI no longer collapses to zero on sparse objectives (and we stop falling back to UCB every phase).

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from faro.agents.bo_optimization import (
    BOptGPAX,
    BO_Parameter,
    BO_Covariate,
    BO_Objective,
    StandardScalerBounds,
)

# Path to the completed live run
RUN_PATH = r"E:\Alex\2026-04-10_bo_erk_oscillation"
CHECKPOINT_DIR = os.path.join(RUN_PATH, "checkpoints")

# Load every per-phase checkpoint
ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "bo_results_phase_*.parquet")))
print(f"Found {len(ckpts)} checkpoints")

# The final checkpoint has all observations
df_all = pd.read_parquet(ckpts[-1])
print(f"Total observations: {len(df_all)}")
print(df_all.head())

## Rebuild the BO agent offline

Reconstruct an agent with the same parameter/covariate definitions used in the live run, then feed it the collected observations.

In [ ]:
# Same definitions as the live notebook
bo_params = [
    BO_Parameter(name="stim_exposure", bounds=(50.0, 500.0), spacing=25.0),
    BO_Parameter(name="ramp", bounds=(0.0, 50.0), spacing=5.0),
]
bo_covariates = [
    BO_Covariate(name="n_cells"),
    BO_Covariate(name="optortk_expression"),
    BO_Covariate(name="baseline_cnr"),
]
bo_objective = BO_Objective(name="frac_oscillating", goal="maximize")


# Minimal agent subclass that doesn't touch the microscope
class _ReplayAgent(BOptGPAX):
    def _create_events_for_cycle(self, parameters):
        raise NotImplementedError("offline replay")

    def _preprocess_results(self, tracks):
        return pd.DataFrame()


agent = _ReplayAgent(
    storage_path=os.path.join(RUN_PATH, "replay_fixed_gpax"),
    parameters_to_optimize=bo_params,
    objective_metric=bo_objective,
    bo_covariates=bo_covariates,
    n_iterations=10,
    n_conditions_per_iter=2,
    n_initial_phases=2,
    acquisition_function="ei",
    n_cov_samples=20,
    ei_xi=0.2,
    ei_xi_final=0.01,
    ei_xi_decay_fraction=0.7,
    verbose=False,
)
os.makedirs(agent.storage_path, exist_ok=True)
agent.df_results = df_all.copy()
print(f"Agent ready with {len(agent.df_results)} observations")

## Ask the fixed BO what it would pick next

This fits the GP on the full observation set, computes the robust EI (with the GP-predicted best-mean reference), and returns the top `n_conditions_per_iter` picks.

In [ ]:
# Pretend we're at iteration N (so xi decay uses the final value)
agent.iteration = agent.n_iterations - 1

picks = agent._select_batch_parameters(
    agent.df_results, n_conditions=agent.n_conditions_per_iter
)
print("Next conditions the fixed BO would measure:")
for i, p in enumerate(picks):
    print(f"  {i}: {p}")

## Visualise the GP landscape + acquisition

In [ ]:
import jax.numpy as jnp

ctx = agent._last_plot_context
gp_model = ctx["gp_model"]
x_scaler = ctx["x_scaler"]
y_scaler = ctx["y_scaler"]
rng_pred = ctx["rng_key_predict"]
acq_values = np.asarray(ctx["acq_values_total"])
acq_name = ctx["acquisition_used"].upper()

# Build control grid
x_total = agent.x_total_linespace.copy()
unique_x1 = np.unique(x_total[:, 0])
unique_x2 = np.unique(x_total[:, 1])
n_ctrl = len(unique_x1) * len(unique_x2)
ctrl_grid = np.array([[x1, x2] for x1 in unique_x1 for x2 in unique_x2])

# Covariate samples (joint)
cov_cols = [c.name for c in agent.bo_covariates]
cov_vals = df_all[cov_cols].to_numpy(dtype=float)
n_cov_samples = 50
rng = np.random.default_rng(0)
cov_samples = cov_vals[rng.integers(0, cov_vals.shape[0], size=n_cov_samples)]

x_full = np.hstack(
    [
        np.repeat(ctrl_grid, n_cov_samples, axis=0),
        np.tile(cov_samples, (n_ctrl, 1)),
    ]
)
x_full_scaled = x_scaler.transform(x_full)

y_pred_scaled, _ = gp_model.predict_in_batches(
    rng_pred,
    x_full_scaled,
    batch_size=1000,
    n=agent.ei_num_samples,
    noiseless=True,
)
y_pred = y_scaler.inverse_transform(np.asarray(y_pred_scaled).reshape(-1, 1)).flatten()
y_pred_marg = y_pred.reshape(n_ctrl, n_cov_samples).mean(axis=1)

X_mesh, Y_mesh = np.meshgrid(unique_x1, unique_x2, indexing="ij")
y_2d = y_pred_marg.reshape(len(unique_x1), len(unique_x2))

# Pad acq to full grid
if len(acq_values) != n_ctrl:
    acq_full = np.zeros(n_ctrl)
    for j, pt in enumerate(ctx["x_unmeasured_at_computation"]):
        diffs = np.abs(x_total - pt).sum(axis=1)
        acq_full[int(np.argmin(diffs))] = (
            float(acq_values[j]) if j < len(acq_values) else 0.0
        )
    acq_values = acq_full
acq_2d = acq_values.reshape(len(unique_x1), len(unique_x2))

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=150)
fig.suptitle("Replay with fixed gpax BO (offline)", fontsize=13)

# 1. Measured points
sc = axes[0].scatter(
    df_all["stim_exposure"] + rng.normal(0, 3, len(df_all)),
    df_all["ramp"] + rng.normal(0, 0.5, len(df_all)),
    c=df_all["frac_oscillating"],
    cmap="viridis",
    s=30,
    edgecolors="k",
    linewidths=0.3,
    alpha=0.8,
)
axes[0].set_xlabel("stim_exposure (ms)")
axes[0].set_ylabel("ramp (ms/frame)")
axes[0].set_title("Measured frac_oscillating")
fig.colorbar(sc, ax=axes[0], label="frac_oscillating")

# 2. GP landscape
im1 = axes[1].pcolormesh(X_mesh, Y_mesh, y_2d, cmap="viridis", shading="auto")
fig.colorbar(im1, ax=axes[1], label="predicted frac_osc")
axes[1].scatter(
    df_all["stim_exposure"],
    df_all["ramp"],
    c="white",
    s=15,
    alpha=0.6,
    marker="x",
    linewidths=0.8,
)
axes[1].set_xlabel("stim_exposure (ms)")
axes[1].set_ylabel("ramp (ms/frame)")
axes[1].set_title("GP predicted landscape (marginalised)")

# 3. Acquisition + picks
im2 = axes[2].pcolormesh(X_mesh, Y_mesh, acq_2d, cmap="inferno", shading="auto")
fig.colorbar(im2, ax=axes[2], label=f"{acq_name} acquisition")
axes[2].scatter(
    df_all["stim_exposure"],
    df_all["ramp"],
    c="white",
    s=15,
    alpha=0.6,
    marker="x",
    linewidths=0.8,
)
picks_arr = np.array([[p["stim_exposure"], p["ramp"]] for p in picks])
axes[2].scatter(
    picks_arr[:, 0],
    picks_arr[:, 1],
    c="cyan",
    s=120,
    marker="X",
    edgecolors="k",
    linewidths=1.0,
    zorder=10,
    label="next conditions",
)
axes[2].legend(loc="upper right", fontsize=8)
axes[2].set_xlabel("stim_exposure (ms)")
axes[2].set_ylabel("ramp (ms/frame)")
axes[2].set_title(f"Acquisition {acq_name} (marginalised)")

plt.tight_layout()
plt.show()